In [72]:
from pathlib import Path
from datetime import date

import pandas as pd
import numpy as np
from numpy.ma.core import diag

from maps import zsdkap_new_columns_names, zsdkap_dtypes, vbap_new_columns_names, mb5td_new_columns_names, mb5td_dtypes, zek103_new_columns_names, zek103_dtypes, mb52_dtypes, mb52_new_columns_names, cohv_dtypes, cohv_new_columns_names
from py_rfc_methods import get_delivery_plants_and_special_stock_indicators_df
from helper_functions import clean_number

In [73]:
SAP_SYSTEM = "P11_SSO"
STORAGE_LOCATIONS = ['0004', '0005']
SUPPLYING_PLANT = '2101'

DATES_RANGE_FILTER = {
    '2101': '2026-07-21', #
    '0301': '2026-07-23', #
    '1201': '2026-07-28', # śr --> wt ; pt --> śr
    '3701': '2026-07-23', # wt --> czw , pt --> śr
}

In [74]:
source_files_dir = Path(r"P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\18_MISSING_STOCK_ORDERS_IDENTIFIER")
output_files_dir = Path(r"P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\18_MISSING_STOCK_ORDERS_IDENTIFIER\output")

# 1. Define filenames in ONE place using a dictionary
source_file_names = {
    "zsdkap": "ZSDKAP.csv",
    "mb5td": "MB5TD.XLSX",
    "zek103": "ZEK103.XLSX",
    "mb52": "MB52.XLSX",
    "cohv": "COHV.XLSX"
}

output_file_names = {
    "df_zar": "df_ZAR.xlsx",
    "df_zri": "df_ZRI.xlsx",
    "df_zrv": "df_ZRV.xlsx",
    "df": "df.xlsx",
}

source_files = {key: source_files_dir / name for key, name in source_file_names.items()}
output_files = {key: output_files_dir / name for key, name in output_file_names.items()}

In [75]:
# product_names = ('ZAR')
# mrp_controllers = ('L2B', 'L2R', 'LI2', 'LI4', 'LI7')

product_names = ('ZRE_M', 'ZRE M', 'ZRV_M', 'ZRV M')
mrp_controllers = ('L2E', 'L2V', 'LI1', 'LI3')
# #
# product_names = ('ZRI', )
# mrp_controllers = ('L2I', )

In [76]:
zsdkap = pd.read_csv(source_files["zsdkap"], dtype=zsdkap_dtypes, sep=';', encoding='MacRoman')

In [77]:
df = zsdkap.copy()

In [78]:
df = df.rename(columns=zsdkap_new_columns_names)
df = df[(df['mrp_controller'].isin(mrp_controllers)) & (df['mat_description'].str.startswith(product_names))].reset_index()

In [79]:
df['orders_quantity'] = df['orders_quantity'].apply(clean_number)
df['orders_quantity'] = df['orders_quantity'].astype(int)
df['customer_order_position'] = df['customer_order_position'].str.zfill(6)
df['dispatch_date'] = pd.to_datetime(df['dispatch_date'], dayfirst=True, errors='coerce')
df['creation_date'] = pd.to_datetime(df['creation_date'], dayfirst=True, errors='coerce')

In [80]:
columns_to_drop = ['Land', 'Postleitzahl', 'Ort', 'Strasse', 'Name', 'UPS', 'Materialgruppe', 'Materialgruppenbezeichnung', 'Bestellnummer', 'Versandbedingung', 'Uhrzeiten', 'Kopfnotiz 3', 'Kopfnotiz 4', 'Gewicht', 'Volumen', 'Lieferpriorität', 'Route', 'Transportzone WE', 'Wskaźnik przetw. specj.', 'ID kontenera', 'Spedition']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

In [81]:
delivery_plants = get_delivery_plants_and_special_stock_indicators_df(SAP_SYSTEM, df['customer_order_number'].tolist(), 1000, 100)
delivery_plants = delivery_plants.rename(columns=vbap_new_columns_names)

2026-07-21 08:10:05 | INFO     | SAP_CONN | Buduję parametry dla systemu P11_SSO: {'mshost': 'rffsp11s.sap.roto-frank.com', 'msserv': 'sapmsP11', 'sysid': 'P11', 'group': 'ROTO_FRANK', 'client': '151', 'lang': 'PL', 'snc_mode': '1', 'snc_qop': '8', 'snc_partnername': ' ', 'timeout': '300'}
2026-07-21 08:10:06 | INFO     | SAP_CONN | Połączenie z SAP nawiązane (system=P11_SSO).
2026-07-21 08:10:06 | INFO     | SAP_CONN | Połączenie SAP zamknięte (system=P11_SSO).


In [82]:
df = pd.merge(df, delivery_plants, how='left', on=['customer_order_number', 'customer_order_position'])
df.dropna(subset=['delivery_plant'], inplace=True)

In [83]:
df = df[df['special_stock_indicator'] != 'E']

In [84]:
zsdkap_customer_orders = df.copy()

In [85]:
# # Newest dispatch_date first, oldest creation_date second
# df_sorted = df.sort_values(by=['dispatch_date', 'creation_date'], ascending=[False, True])

In [86]:
df_grouped = df.groupby(['delivery_plant', 'mat_number', 'dispatch_date'], as_index=False).agg({
    'orders_quantity': 'sum',
    'mat_description': 'first',
})

In [87]:
df_grouped.head(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description
0,0301,843703,2026-07-20,1,ZRV M 065/118 R6R8 AV03
1,0301,843703,2026-07-21,1,ZRV M 065/118 R6R8 AV03
2,0301,843703,2026-07-28,1,ZRV M 065/118 R6R8 AV03
3,0301,844065,2026-07-24,1,ZRV M 094/140 R6R8 AV05
4,0301,844065,2026-07-30,1,ZRV M 094/140 R6R8 AV05


In [88]:
sap_list = df['mat_number'].tolist()

In [89]:
zek103_df = pd.read_excel(source_files['zek103'], dtype=zek103_dtypes)
zek103_df.rename(columns=zek103_new_columns_names, inplace=True)

In [90]:
zek103_df['po_delivery_date'] = pd.to_datetime(zek103_df['po_delivery_date'], dayfirst=True, errors='coerce')
zek103_df['po_dispatch_date'] = pd.to_datetime(zek103_df['po_dispatch_date'], dayfirst=True, errors='coerce')
zek103_df = zek103_df[zek103_df['mat_number'].isin(sap_list)]

In [91]:
cols_to_drop = ['Szuk. ciąg', 'LFT', 'Best-Mg', 'Bestät. Menge', 'ME', 'Best. Liefdat.']
zek103_df = zek103_df.drop(columns=[col for col in cols_to_drop if col in zek103_df.columns])

In [92]:
zek103_df.head(5)

,purchase_order_number,purchase_order_position,mat_number,mat_description_zek103,po_delivery_date,po_quantity,plant,po_dispatch_date,customer_order_number,customer_order_position
84,5215368,10,884381,ZRV R4/R7x 05/07 M W_ V22,2021-06-01,0.0,1201,2021-05-28,<NA>,0
92,5702097,10,884889,ZRV M 114/118 R4R7 WV06,2023-07-04,0.0,1201,2023-06-30,<NA>,0
93,5703827,10,884889,ZRV M 114/118 R4R7 WV06,2023-07-04,0.0,1201,2023-06-30,<NA>,0
94,5722425,40,884644,ZRV M 074/118 R4R7 WV01,2023-08-02,0.0,3701,2023-07-31,<NA>,0
95,5736706,30,884649,ZRV M 074/118 R4R7 WV06,2023-08-30,0.0,3701,2023-08-28,<NA>,0


In [93]:
# Drop purchase orders linked to special customer requirements
zek103_df = zek103_df[zek103_df['customer_order_number'].isna()]

In [94]:
zek103_2_df = zek103_df.copy() # I copy ZEK103 df to get quantities which will be dispatched from production plant

In [95]:
zek103_df = zek103_df.groupby(['po_delivery_date', 'plant', 'mat_number'], as_index=False).agg({
    'po_quantity': 'sum',
    'po_dispatch_date': 'first',
    'mat_description_zek103': 'first',

})

In [96]:
zsdkap_zek103_merged_df = pd.merge(df_grouped, zek103_df, how='outer', left_on=['delivery_plant', 'mat_number', 'dispatch_date'], right_on=['plant', 'mat_number', 'po_delivery_date'])

In [97]:
zsdkap_zek103_merged_df['delivery_plant'] = zsdkap_zek103_merged_df['delivery_plant'].combine_first(zsdkap_zek103_merged_df['plant'])
zsdkap_zek103_merged_df['dispatch_date'] = zsdkap_zek103_merged_df['dispatch_date'].combine_first(zsdkap_zek103_merged_df['po_delivery_date'])
zsdkap_zek103_merged_df['mat_description'] = zsdkap_zek103_merged_df['mat_description'].combine_first(zsdkap_zek103_merged_df['mat_description_zek103'])

In [98]:
zsdkap_zek103_merged_df['orders_quantity'] = zsdkap_zek103_merged_df['orders_quantity'].fillna(0)
zsdkap_zek103_merged_df['po_quantity'] = zsdkap_zek103_merged_df['po_quantity'].fillna(0)

In [99]:
zsdkap_zek103_merged_df = zsdkap_zek103_merged_df.drop(columns=['plant', 'po_delivery_date', 'mat_description_zek103'])

In [100]:
zsdkap_zek103_merged_df.head(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date
0,0301,843703,2026-07-20,1.0,ZRV M 065/118 R6R8 AV03,0.0,NaT
1,0301,843703,2026-07-21,1.0,ZRV M 065/118 R6R8 AV03,0.0,NaT
2,0301,843703,2026-07-27,0.0,ZRV M 065/118 R6R8 AV03,5.0,2026-07-23
3,0301,843703,2026-07-28,1.0,ZRV M 065/118 R6R8 AV03,0.0,NaT
4,0301,844065,2026-07-23,0.0,ZRV M 094/140 R6R8 AV05,2.0,2026-07-21


In [101]:
mb52_df = pd.read_excel(source_files['mb52'], dtype=mb52_dtypes)
mb52_df.rename(columns=mb52_new_columns_names, inplace=True)
mb52_df = mb52_df[(mb52_df['mat_number'].isin(sap_list)) & (mb52_df['customer_order_number'].isna()) & (mb52_df['storage_location'].isin(STORAGE_LOCATIONS))]

In [102]:
mb52_df = mb52_df.groupby(['plant', 'mat_number'], as_index=False).agg({
    'stock_quantity': 'sum',
})

In [103]:
mb52_df.head()

,plant,mat_number,stock_quantity
0,0301,843161,9.0
1,0301,843703,4.0
2,0301,844065,1.0
3,0301,844241,8.0
4,0301,848325,4.0


In [104]:
zsdkap_zek103_mb52_merged_df = pd.merge(zsdkap_zek103_merged_df, mb52_df, left_on=['delivery_plant', 'mat_number'], right_on=['plant', 'mat_number']).drop(columns=['plant'])

In [105]:
zsdkap_zek103_mb52_merged_df.head(10)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity
0,0301,843703,2026-07-20,1.0,ZRV M 065/118 R6R8 AV03,0.0,NaT,4.0
1,0301,843703,2026-07-21,1.0,ZRV M 065/118 R6R8 AV03,0.0,NaT,4.0
2,0301,843703,2026-07-27,0.0,ZRV M 065/118 R6R8 AV03,5.0,2026-07-23,4.0
3,0301,843703,2026-07-28,1.0,ZRV M 065/118 R6R8 AV03,0.0,NaT,4.0
4,0301,844065,2026-07-23,0.0,ZRV M 094/140 R6R8 AV05,2.0,2026-07-21,1.0
5,0301,844065,2026-07-24,1.0,ZRV M 094/140 R6R8 AV05,0.0,NaT,1.0
6,0301,844065,2026-07-30,1.0,ZRV M 094/140 R6R8 AV05,0.0,NaT,1.0
7,0301,844241,2026-07-28,1.0,ZRV M 114/140 R6R8 AV01,0.0,NaT,8.0
8,0301,848325,2026-07-21,2.0,ZRE M 094/118 R6R8 AR01,0.0,NaT,4.0
9,0301,879921,2026-07-22,1.0,ZRE M 074/118 R6R8 WR03,0.0,NaT,7.0


In [106]:
cohv_df = pd.read_excel(source_files['cohv'], dtype=cohv_dtypes)
cohv_df = cohv_df.rename(columns=cohv_new_columns_names)
cohv_df['production_date'] = pd.to_datetime(cohv_df['production_date'], dayfirst=True, errors='coerce')

In [107]:
cohv_df = cohv_df[cohv_df['mat_number'].isin(sap_list)]
cohv_df = cohv_df.groupby(['production_plant', 'mat_number', 'production_date'], as_index=False).agg({
    'production_quantity': 'sum',
})

In [108]:
cohv_df.head(5)

,production_plant,mat_number,production_date,production_quantity
0,2101,841487,2026-07-23,10
1,2101,842868,2026-07-23,10
2,2101,843108,2026-07-22,5
3,2101,843131,2026-07-23,10
4,2101,843135,2026-07-21,5


In [109]:
zsdkap_zek103_mb52_cohv_merged_df = (pd.merge(zsdkap_zek103_mb52_merged_df, cohv_df, left_on=['delivery_plant', 'mat_number', 'dispatch_date'], right_on=['production_plant', 'mat_number', 'production_date'], how='outer'))
zsdkap_zek103_mb52_cohv_merged_df['dispatch_date'] = zsdkap_zek103_mb52_cohv_merged_df['dispatch_date'].combine_first(zsdkap_zek103_mb52_cohv_merged_df['production_date'])
zsdkap_zek103_mb52_cohv_merged_df = zsdkap_zek103_mb52_cohv_merged_df.drop(columns=['production_date', 'production_plant'])

In [110]:
zsdkap_zek103_mb52_cohv_merged_df['production_quantity'] = zsdkap_zek103_mb52_cohv_merged_df['production_quantity'].fillna(0)
zsdkap_zek103_mb52_cohv_merged_df['delivery_plant'] = zsdkap_zek103_mb52_cohv_merged_df['delivery_plant'].fillna(SUPPLYING_PLANT)

In [111]:
mb5td_df = pd.read_excel(source_files["mb5td"], dtype=mb5td_dtypes)
mb5td_df.rename(columns=mb5td_new_columns_names, inplace=True)
mb5td_df = mb5td_df[mb5td_df['mat_number'].isin(sap_list)]

In [112]:
mb5td_df = mb5td_df[mb5td_df['special_stock_indicator'].isna()]

In [113]:
mb5td_df = mb5td_df[['mat_number', 'supplying_plant', 'purchase_order_number', 'purchase_order_position', 'transit_quantity']]

In [114]:
mb5td_df.head()

,mat_number,supplying_plant,purchase_order_number,purchase_order_position,transit_quantity
243,885004,2101,4500138815,10,6.0
249,882939,2101,4500138836,10,5.0
252,843498,2101,4500138836,40,10.0
258,884885,2101,4500138837,60,5.0
259,884884,2101,4500138838,10,5.0


In [115]:
zek103_2_df.head(5)

,purchase_order_number,purchase_order_position,mat_number,mat_description_zek103,po_delivery_date,po_quantity,plant,po_dispatch_date,customer_order_number,customer_order_position
84,5215368,10,884381,ZRV R4/R7x 05/07 M W_ V22,2021-06-01,0.0,1201,2021-05-28,<NA>,0
92,5702097,10,884889,ZRV M 114/118 R4R7 WV06,2023-07-04,0.0,1201,2023-06-30,<NA>,0
93,5703827,10,884889,ZRV M 114/118 R4R7 WV06,2023-07-04,0.0,1201,2023-06-30,<NA>,0
94,5722425,40,884644,ZRV M 074/118 R4R7 WV01,2023-08-02,0.0,3701,2023-07-31,<NA>,0
95,5736706,30,884649,ZRV M 074/118 R4R7 WV06,2023-08-30,0.0,3701,2023-08-28,<NA>,0


In [116]:
zek103_2_mb5td_merged_df = pd.merge(zek103_2_df, mb5td_df, on=['mat_number', 'purchase_order_number', 'purchase_order_position'], how='left')

In [117]:
zek103_2_mb5td_merged_df['transit_quantity'] = zek103_2_mb5td_merged_df['transit_quantity'].fillna(0)
zek103_2_mb5td_merged_df['supplying_plant'] = zek103_2_mb5td_merged_df['supplying_plant'].fillna(SUPPLYING_PLANT)
zek103_2_mb5td_merged_df['dispatched_quantity'] = zek103_2_mb5td_merged_df['po_quantity'] - zek103_2_mb5td_merged_df['transit_quantity']

In [118]:
zek103_2_mb5td_merged_df.head(10)

,purchase_order_number,purchase_order_position,mat_number,mat_description_zek103,po_delivery_date,po_quantity,plant,po_dispatch_date,customer_order_number,customer_order_position,supplying_plant,transit_quantity,dispatched_quantity
0,5215368,10,884381,ZRV R4/R7x 05/07 M W_ V22,2021-06-01,0.0,1201,2021-05-28,<NA>,0,2101,0.0,0.0
1,5702097,10,884889,ZRV M 114/118 R4R7 WV06,2023-07-04,0.0,1201,2023-06-30,<NA>,0,2101,0.0,0.0
2,5703827,10,884889,ZRV M 114/118 R4R7 WV06,2023-07-04,0.0,1201,2023-06-30,<NA>,0,2101,0.0,0.0
3,5722425,40,884644,ZRV M 074/118 R4R7 WV01,2023-08-02,0.0,3701,2023-07-31,<NA>,0,2101,0.0,0.0
4,5736706,30,884649,ZRV M 074/118 R4R7 WV06,2023-08-30,0.0,3701,2023-08-28,<NA>,0,2101,0.0,0.0
5,5904159,20,883834,ZRV M 114/118 Qx__ WV03,2024-08-13,0.0,1201,2024-08-07,<NA>,0,2101,0.0,0.0
6,4500104559,60,884644,ZRV M 074/118 R4R7 WV01,2025-11-12,0.0,3701,2025-11-07,<NA>,0,2101,0.0,0.0
7,4500137650,30,885729,ZRV M 114/118 R6R8 WV06,2026-07-27,5.0,0301,2026-07-23,<NA>,0,2101,0.0,5.0
8,4500137921,10,884494,ZRV M 065/118 R4R7 WV01,2026-07-27,5.0,0301,2026-07-23,<NA>,0,2101,0.0,5.0
9,4500138217,40,887198,ZRV M 105/112 8AR1 WV05,2026-07-27,5.0,0301,2026-07-23,<NA>,0,2101,0.0,5.0


In [119]:
dispatches_df = zek103_2_mb5td_merged_df.groupby(['mat_number', 'supplying_plant', 'po_dispatch_date'], as_index=False).agg({
    'dispatched_quantity': 'sum',
    # 'mat_description_zek103': 'first',
})

In [120]:
dispatches_df = dispatches_df[dispatches_df['dispatched_quantity'] > 0]

In [121]:
dispatches_df.head()

,mat_number,supplying_plant,po_dispatch_date,dispatched_quantity
0,841487,2101,2026-07-24,10.0
1,842868,2101,2026-07-24,10.0
3,843108,2101,2026-07-22,5.0
4,843108,2101,2026-07-24,5.0
5,843375,2101,2026-07-24,5.0


In [122]:
final_df = zsdkap_zek103_mb52_cohv_merged_df.copy()

In [123]:
final_df[final_df['delivery_plant'] == '2101'].head(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity,production_quantity
295,2101,5000276,2026-07-20,18.0,ZRV M 054/078 R4R7 AV22 KF,0.0,NaT,410.0,0.0
296,2101,5000276,2026-08-13,48.0,ZRV M 054/078 R4R7 AV22 KF,0.0,NaT,410.0,0.0
297,2101,5000277,2026-07-20,24.0,ZRV M 078/098 R4R7 AV22 KF,0.0,NaT,417.0,0.0
298,2101,5000277,2026-08-13,48.0,ZRV M 078/098 R4R7 AV22 KF,0.0,NaT,417.0,0.0
299,2101,5000278,2026-07-20,31.0,ZRV M 114/118 R4R7 AV22 KF,0.0,NaT,255.0,0.0


In [124]:
final_df = (pd.merge(final_df, dispatches_df, left_on=['delivery_plant', 'mat_number', 'dispatch_date'], right_on=['supplying_plant', 'mat_number', 'po_dispatch_date'], suffixes=('', '_disp'), how='outer'))
final_df['dispatch_date'] = final_df['dispatch_date'].combine_first(final_df['po_dispatch_date_disp'])
final_df['delivery_plant'] = final_df['delivery_plant'].combine_first(final_df['supplying_plant'])
final_df = final_df.drop(columns=['po_dispatch_date_disp', 'supplying_plant'])

final_df['orders_quantity'] = final_df['orders_quantity'].fillna(0)
final_df['production_quantity'] = final_df['production_quantity'].fillna(0)
final_df['dispatched_quantity'] = final_df['dispatched_quantity'].fillna(0)
final_df['po_quantity'] = final_df['po_quantity'].fillna(0)

# final_df['mat_description'] = final_df.groupby(['mat_number', 'delivery_plant'])['mat_description'].ffill().bfill() # This line fulfills mat description from other rows with the same mat number
# final_df['stock_quantity'] = final_df.groupby(['mat_number', 'delivery_plant'])['stock_quantity'].ffill().bfill()   # This line fulfills stock quantity from other rows with the same mat number

# For material description
final_df['mat_description'] = (
    final_df.groupby(['mat_number', 'delivery_plant'])['mat_description']
    .transform(lambda x: x.ffill().bfill())
).infer_objects(copy=False)

# For stock quantity
final_df['stock_quantity'] = (
    final_df.groupby(['mat_number', 'delivery_plant'])['stock_quantity']
    .transform(lambda x: x.ffill().bfill())
).infer_objects(copy=False)

C:\Users\daniewskik\AppData\Local\Temp\ipykernel_26776\3282838051.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda x: x.ffill().bfill())


In [125]:
final_df.sample(5)

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity,production_quantity,dispatched_quantity
232,1201,843498,2026-07-22,1.0,ZRV M 078/098 R4R7 AV22,0.0,NaT,8.0,0.0,0.0
261,1201,884885,2026-07-21,0.0,ZRV M 114/118 R4R7 WV02,5.0,2026-07-15,10.0,0.0,0.0
260,1201,884885,2026-07-17,3.0,ZRV M 114/118 R4R7 WV02,0.0,NaT,10.0,0.0,0.0
185,0301,885761,2026-07-24,0.0,ZRV M 114/140 R6R8 WV22,5.0,2026-07-22,2.0,0.0,0.0
203,0301,887284,2026-07-20,3.0,ZRV M 105/145 8AR1 WV01,0.0,NaT,5.0,0.0,0.0


In [126]:
# Sort the rows by plant, material number, and date
final_df = final_df.sort_values(by=['delivery_plant', 'mat_number', 'dispatch_date']).reset_index(drop=True)

In [127]:
# Calculate the running totals (cumulative sums) for orders and POs within each group
final_df['cum_orders'] = final_df.groupby(['delivery_plant', 'mat_number'])['orders_quantity'].cumsum()
final_df['cum_po'] = final_df.groupby(['delivery_plant', 'mat_number'])['po_quantity'].cumsum()
final_df['cum_prod'] = final_df.groupby(['delivery_plant', 'mat_number'])['production_quantity'].cumsum()
final_df['cum_dispatches'] = final_df.groupby(['delivery_plant', 'mat_number'])['dispatched_quantity'].cumsum()

# Compute the final stock left column
final_df['stock_left'] = final_df['stock_quantity'] + final_df['cum_po'] - final_df['cum_orders'] + final_df['cum_prod'] - final_df['cum_dispatches']

In [128]:
# 6. Drop the temporary cumulative columns
final_df = final_df.drop(columns=['cum_orders', 'cum_po', 'cum_prod', 'cum_dispatches'])

In [129]:
final_df.head()

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity,production_quantity,dispatched_quantity,stock_left
0,0301,843703,2026-07-20,1.0,ZRV M 065/118 R6R8 AV03,0.0,NaT,4.0,0.0,0.0,3.0
1,0301,843703,2026-07-21,1.0,ZRV M 065/118 R6R8 AV03,0.0,NaT,4.0,0.0,0.0,2.0
2,0301,843703,2026-07-27,0.0,ZRV M 065/118 R6R8 AV03,5.0,2026-07-23,4.0,0.0,0.0,7.0
3,0301,843703,2026-07-28,1.0,ZRV M 065/118 R6R8 AV03,0.0,NaT,4.0,0.0,0.0,6.0
4,0301,844065,2026-07-23,0.0,ZRV M 094/140 R6R8 AV05,2.0,2026-07-21,1.0,0.0,0.0,3.0


In [130]:
# 2. Map the dictionary to create a temporary "boundary date" series for each row
# We also convert the dictionary values to datetime on the fly
boundary_dates = final_df['delivery_plant'].map(DATES_RANGE_FILTER)
boundary_dates = pd.to_datetime(boundary_dates)

# 3. Keep rows where dispatch_date is less than or equal to the boundary date.
# (If a plant isn't in your dictionary, we keep it by using .isna())
filtered_final_df = final_df[
    (final_df['dispatch_date'] <= boundary_dates) | (boundary_dates.isna())
]

In [131]:
filtered_final_df.to_excel(output_files['df_zrv'], index=False)

In [132]:
filtered_final_df[filtered_final_df['stock_left'] < 0]

,delivery_plant,mat_number,dispatch_date,orders_quantity,mat_description,po_quantity,po_dispatch_date,stock_quantity,production_quantity,dispatched_quantity,stock_left
35,0301,884644,2026-07-22,5.0,ZRV M 074/118 R4R7 WV01,0.0,NaT,7.0,0.0,0.0,-1.0
71,0301,885188,2026-07-23,7.0,ZRV M 065/098 R6R8 WV05,0.0,NaT,5.0,0.0,0.0,-1.0


### Delayed orders identification

In [133]:
shortages_df = filtered_final_df[filtered_final_df['stock_left'] < 0][['delivery_plant', 'mat_number', 'mat_description', 'dispatch_date', 'stock_left']]

In [134]:
shortages_df.head()

,delivery_plant,mat_number,mat_description,dispatch_date,stock_left
35,0301,884644,ZRV M 074/118 R4R7 WV01,2026-07-22,-1.0
71,0301,885188,ZRV M 065/098 R6R8 WV05,2026-07-23,-1.0


In [135]:
zsdkap_customer_orders =  zsdkap_customer_orders[['delivery_plant', 'mat_number', 'mat_description', 'orders_quantity', 'creation_date', 'dispatch_date', 'customer_order_number', 'customer_order_position']]

In [136]:
zsdkap_customer_orders = zsdkap_customer_orders.sort_values(by=['delivery_plant', 'mat_number', 'creation_date', 'dispatch_date'], ascending=[True, True, False, True]).reset_index(drop=True)

In [137]:
zsdkap_customer_orders['cum_orders'] = zsdkap_customer_orders.groupby(['delivery_plant', 'mat_number', 'dispatch_date'])['orders_quantity'].cumsum()

In [138]:
zsdkap_customer_orders.head(20)

,delivery_plant,mat_number,mat_description,orders_quantity,creation_date,dispatch_date,customer_order_number,customer_order_position,cum_orders
0,0301,843703,ZRV M 065/118 R6R8 AV03,1,2026-07-17,2026-07-28,3110018227,000040,1
1,0301,843703,ZRV M 065/118 R6R8 AV03,1,2026-07-15,2026-07-20,0213941071,000010,1
2,0301,843703,ZRV M 065/118 R6R8 AV03,1,2026-07-15,2026-07-21,0213940870,000010,1
3,0301,844065,ZRV M 094/140 R6R8 AV05,1,2026-07-17,2026-07-30,2710153371,000010,1
4,0301,844065,ZRV M 094/140 R6R8 AV05,1,2026-07-13,2026-07-24,0213939197,000010,1
5,0301,844241,ZRV M 114/140 R6R8 AV01,1,2026-07-16,2026-07-28,1100013844,000070,1
6,0301,844393,ZRV M 134/140 R6R8 AV03,3,2026-07-16,2026-07-27,0213941844,000060,3
7,0301,848325,ZRE M 094/118 R6R8 AR01,2,2026-07-14,2026-07-21,0213940267,000010,2
8,0301,879921,ZRE M 074/118 R6R8 WR03,1,2026-07-17,2026-07-23,3110018253,000030,1
9,0301,879921,ZRE M 074/118 R6R8 WR03,1,2026-07-16,2026-07-22,0213941787,000010,1


In [139]:
# # Zamiast pustego DataFrame, tworzymy pustą listę do zbierania wyników
# delayed_orders_list = []
#
# for row in shortages_df.iterrows():
#     row_plant = row[1]['delivery_plant']
#     row_sap = row[1]['mat_number']
#     row_date = row[1]['dispatch_date']
#     row_shortage = row[1]['stock_left'] * -1
#
#     print(row_plant, row_date, row_shortage)
#     # TODO: Dopracować ostatni warunek
#     temp_df = zsdkap_customer_orders[
#         (zsdkap_customer_orders['delivery_plant'] == row_plant) &
#         (zsdkap_customer_orders['mat_number'] == row_sap) &
#         (zsdkap_customer_orders['dispatch_date'] == row_date)
#     ].copy()
#
#     temp_df['shortage'] = row_shortage
#
#     # Po prostu wrzucamy wynik do listy (jeśli nie jest pusty)
#     if not temp_df.empty:
#         delayed_orders_list.append(temp_df)
#
# # Po zakończeniu pętli łączymy wszystko w jeden DataFrame
# if delayed_orders_list:
#     delayed_orders = pd.concat(delayed_orders_list, ignore_index=True)
# else:
#     # Zabezpieczenie na wypadek, gdyby pętla nic nie znalazła
#     delayed_orders = pd.DataFrame(columns=zsdkap_customer_orders.columns)

# 1. Przygotowujemy shortages_df dodając kolumnę z wielkością braku (dodatnią)
shortages_temp = shortages_df[['delivery_plant', 'mat_number', 'dispatch_date', 'stock_left']].copy()
shortages_temp['shortage'] = shortages_temp['stock_left'] * -1
shortages_temp = shortages_temp.drop(columns=['stock_left'])

# 2. Łączymy wektorowo zamiast pętli for
delayed_orders = pd.merge(
    zsdkap_customer_orders,
    shortages_temp,
    on=['delivery_plant', 'mat_number', 'dispatch_date'],
    how='inner' # zostawiamy tylko te wiersze, które mają niedobory
)

delayed_orders

,delivery_plant,mat_number,mat_description,orders_quantity,creation_date,dispatch_date,customer_order_number,customer_order_position,cum_orders,shortage
0,0301,884644,ZRV M 074/118 R4R7 WV01,5,2026-07-15,2026-07-22,0213940790,000150,5,1.0
1,0301,885188,ZRV M 065/098 R6R8 WV05,3,2026-07-14,2026-07-23,0213940111,000030,3,1.0
2,0301,885188,ZRV M 065/098 R6R8 WV05,4,2026-07-14,2026-07-23,0213940437,000010,7,1.0


In [140]:
zsdkap_customer_orders.to_excel(output_files['df'], index=False)

In [141]:
delayed_orders['to_be_delayed'] = (
    (delayed_orders['cum_orders'] - delayed_orders['orders_quantity']) < delayed_orders['shortage']
)

In [142]:
delayed_orders

,delivery_plant,mat_number,mat_description,orders_quantity,creation_date,dispatch_date,customer_order_number,customer_order_position,cum_orders,shortage,to_be_delayed
0,0301,884644,ZRV M 074/118 R4R7 WV01,5,2026-07-15,2026-07-22,0213940790,000150,5,1.0,True
1,0301,885188,ZRV M 065/098 R6R8 WV05,3,2026-07-14,2026-07-23,0213940111,000030,3,1.0,True
2,0301,885188,ZRV M 065/098 R6R8 WV05,4,2026-07-14,2026-07-23,0213940437,000010,7,1.0,False
